In [1]:
import sys

print("Python path:", sys.executable)

Python path: C:\Users\DELL\anaconda3\envs\qafza-mlops\python.exe


# Create Delivery Labels

This notebook creates the delivery label by comparing the actual delivery date with the estimated delivery date.

## 1. Load the Prepared Dataset

I load the order-level dataset created in Notebook 1.

In [2]:
import pandas as pd

# Load the prepared dataset
data = pd.read_csv("../artifacts/final_order_dataset.csv")

print("Dataset shape:", data.shape)

Dataset shape: (99441, 27)


## 2. Create the Delivery Label

The label is based on whether the actual delivery date was later than the estimated delivery date.

In [3]:
# Create the delivery label
data["delivery_label"] = (
    pd.to_datetime(data["order_delivered_customer_date"])
    > pd.to_datetime(data["order_estimated_delivery_date"])
).map({
    True: "Late",
    False: "On Time"
})

data[["order_id", "delivery_label"]].head()

,order_id,delivery_label
0,e481f51cbdc54678b7cc49136f2d6af7,On Time
1,53cdb2fc8bc7dce0b6741e2150273451,On Time
2,47770eb9100c2d0c44946d9cf07ec65d,On Time
3,949d5b44dbf5de918fe9c16f97b45f8a,On Time
4,ad21c59c0840e6cb83a9ceb5573f8159,On Time


## 3. Check the Label

I check a few real orders to confirm that the delivery label matches the actual and estimated delivery dates.

In [4]:
# Check a few real orders
data[[
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_label"
]].sample(5, random_state=42)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_label
52263,b9a6c5f5df52c7226ac85aee7524c27f,2018-06-19 12:44:08,2018-07-17,On Time
46645,261e71d2349c713eafa9f3df5972b95d,2018-01-30 11:32:35,2018-02-15,On Time
37546,67b50899f52995848c427e361e10dde3,2018-06-27 13:17:27,2018-07-16,On Time
94756,32733fc014b67ef70fa6039dd8c6ba82,2017-09-25 17:53:23,2017-09-22,Late
14771,39a70e9e9b729b11dee34ac12478597f,2017-08-22 16:45:00,2017-09-12,On Time


## 4. Class Distribution

I examine the distribution of Late and On Time orders to understand the class balance.

In [5]:
# Count each delivery label
data["delivery_label"].value_counts()

delivery_label
On Time    91614
Late        7827
Name: count, dtype: int64

## 5. Keep Orders with Known Delivery Outcomes

The delivery label can only be created for orders with a recorded actual delivery date.

I keep only orders with a known delivery outcome.

In [6]:
# Keep orders with a known delivery date
data = data.dropna(subset=["order_delivered_customer_date"])

print("Labeled orders:", len(data))

Labeled orders: 96476


## 6. Class Imbalance

The class distribution is used to determine whether the dataset has a class imbalance problem.

In [7]:
# Calculate the percentage of each class
data["delivery_label"].value_counts(normalize=True).mul(100)

delivery_label
On Time    91.887101
Late        8.112899
Name: proportion, dtype: float64

## 7. Save the Labeled Dataset

I save the labeled dataset so it can be used by the next notebook.

In [8]:
# Save the labeled dataset
output_path = "../artifacts/labeled_orders.csv"

data.to_csv(output_path, index=False)

print("Labeled dataset saved successfully.")

Labeled dataset saved successfully.
